# Lab 7 — Machine Translation Models

[Open this notebook in Colab](https://colab.research.google.com/github/coastalcph/nlp-course/blob/master/labs/notebooks_2026/lab_7.ipynb), then select **File → Save a copy in Drive**.

This lab demonstrates how to use machine translation models to translate text between languages.

## Learning objectives

By the end of this lab, you should be able to:

1. load pre-trained MT models (Helsinki-NLP and NLLB) with HuggingFace;
2. translate a small dataset of questions from English to Danish;
3. see how MT models handle short inputs such as single words.

## Setup

In [ ]:
%pip install -q "transformers>=4.40" datasets sentencepiece sacremoses

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

## Toy dataset

A small dataset of English questions that we will translate to Danish. The `translate` helper tokenizes all questions together and translates them in a single batch instead of one sentence at a time.

In [ ]:
questions = Dataset.from_dict({"en": [
    "What is machine translation?",
    "How do neural networks work?",
    "What is the capital of Denmark?",
    "When was the University of Copenhagen founded?",
    "How can we improve translation quality?",
]})


def translate(tokenizer, model, dataset):
    inputs = tokenizer(list(dataset["en"]), return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


questions.to_pandas()

## Helsinki-NLP (OPUS-MT)

[Helsinki-NLP](https://huggingface.co/Helsinki-NLP) provides a large family of OPUS-MT models, with a separate small model for each translation direction. The model name tells you the language pair, e.g. `opus-mt-en-da` translates English → Danish and `opus-mt-da-en` translates Danish → English.

In [ ]:
tokenizer_helsinki = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-da")
model_helsinki = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-da")

questions_da_helsinki = translate(tokenizer_helsinki, model_helsinki, questions)
questions_da_helsinki

## NLLB

[`facebook/nllb-200-distilled-600M`](https://huggingface.co/facebook/nllb-200-distilled-600M) is Meta's *No Language Left Behind* model: a single multilingual model that translates between 200 languages.

Because one model covers all languages, we must tell it the source and target language using [FLORES-200 codes](https://github.com/facebookresearch/flores/blob/main/flores200/README.md#languages-in-flores-200) (`eng_Latn`, `dan_Latn`). Without them, the model does not know which language to generate and often copies the input or repeats itself.

In [ ]:
tokenizer_nllb = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="eng_Latn")
model_nllb = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
# NLLB is multilingual, so we force the first generated token to be the target language
model_nllb.generation_config.forced_bos_token_id = tokenizer_nllb.convert_tokens_to_ids("dan_Latn")

questions_da_nllb = translate(tokenizer_nllb, model_nllb, questions)
questions_da_nllb

## Comparison

In [ ]:
pd.DataFrame({
    "English": questions["en"],
    "Helsinki-NLP": questions_da_helsinki,
    "NLLB": questions_da_nllb,
})

## Single words and short phrases

MT models are trained on full sentences, so they expect sentence-like input. With a single word or short phrase there is little context, and the models often treat it as a sentence: they add punctuation (`Tak.`), pick an unexpected word form, or even add words that were not in the input. You can see such behavior in the Outputs below:

In [ ]:
short_inputs = Dataset.from_dict({"en": [
    "hello",
    "cat",
    "beautiful",
    "Copenhagen",
    "thank you",
    "good morning",
]})

pd.DataFrame({
    "English": short_inputs["en"],
    "Helsinki-NLP": translate(tokenizer_helsinki, model_helsinki, short_inputs),
    "NLLB": translate(tokenizer_nllb, model_nllb, short_inputs),
})